In [1]:
# import os
# print("커널이 살아있습니다!")
# print(f"현재 경로: {os.getcwd()}")

커널이 살아있습니다!
현재 경로: /workspace/members/kim


In [1]:
%load_ext autotime

time: 118 μs (started: 2026-05-10 10:01:17 +00:00)


# 1. 라이브러리 임포트, 모델 로드, vLLM 엔진 초기화

In [2]:
print("시작: 라이브러리 임포트 중...")

import multiprocessing
try:
    multiprocessing.set_start_method('spawn', force=True)
    print("- multiprocessing 설정 완료")
except RuntimeError:
    pass

print("- torch 임포트 시작...")
import torch
print("- torch 임포트 완료")

시작: 라이브러리 임포트 중...
- multiprocessing 설정 완료
- torch 임포트 시작...
- torch 임포트 완료
time: 29.3 s (started: 2026-05-10 10:01:24 +00:00)


In [3]:
print("- vllm 임포트 시작 (여기서 시간이 걸릴 수 있습니다)...")
from vllm import LLM, SamplingParams
print("- vllm 임포트 완료")

- vllm 임포트 시작 (여기서 시간이 걸릴 수 있습니다)...
- vllm 임포트 완료
time: 1min 54s (started: 2026-05-10 10:09:16 +00:00)


In [5]:
print("- 기타 라이브러리 임포트 시작")
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor

import subprocess
import os
print("- 라이브러리 임포트 완료")

- 기타 라이브러리 임포트 시작
- 라이브러리 임포트 완료
time: 886 μs (started: 2026-05-10 10:11:30 +00:00)


In [6]:
# --- 설정 및 경로 ---
CHECKPOINT_PATH = "/workspace/models/qwen3-vl-8b-instruct-fp8"

# 1. 모델 로드 (시간이 오래 걸리는 작업)
print("1. Processor 로딩 시작...")
processor = AutoProcessor.from_pretrained(CHECKPOINT_PATH)

print("2. vLLM 엔진 초기화 시작 (이 작업은 네트워크 볼륨 속도에 따라 5~15분 소요됩니다)...")
print("   Tip: 터미널에서 'watch nvidia-smi'로 메모리 증가를 확인하세요.")
llm = LLM(
    model=CHECKPOINT_PATH, trust_remote_code=True, gpu_memory_utilization=0.8, 
    max_model_len=32768, tensor_parallel_size=1, enforce_eager=True          
)

def prepare_inputs(messages, processor):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    _, video_inputs, video_kwargs = process_vision_info(
        messages, return_video_kwargs=True, return_video_metadata=True
    )
    return {'prompt': text, 'multi_modal_data': {'video': video_inputs}, 'mm_processor_kwargs': video_kwargs}

print("✅ vLLM 엔진 내부 준비 완료!")
print("✅ 모델 로드 완료!")

1. Processor 로딩 시작...
2. vLLM 엔진 초기화 시작 (이 작업은 네트워크 볼륨 속도에 따라 5~15분 소요됩니다)...
   Tip: 터미널에서 'watch nvidia-smi'로 메모리 증가를 확인하세요.
INFO 05-10 10:11:35 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/workspace/models/qwen3-vl-8b-instruct-fp8'}
INFO 05-10 10:14:41 [model.py:549] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 05-10 10:14:41 [model.py:1678] Using max model len 32768
INFO 05-10 10:14:44 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-10 10:14:44 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-10 10:14:44 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-10 10:14:44 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during induct

`Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=1891) INFO 05-10 10:14:51 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='/workspace/models/qwen3-vl-8b-instruct-fp8', speculative_config=None, tokenizer='/workspace/models/qwen3-vl-8b-instruct-fp8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=1891) INFO 05-10 10:15:27 [default_loader.py:384] Loading weights took 17.98 seconds
(EngineCore pid=1891) INFO 05-10 10:15:27 [gpu_model_runner.py:4820] Model loading took 10.31 GiB memory and 30.930805 seconds
(EngineCore pid=1891) INFO 05-10 10:15:28 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(EngineCore pid=1891) WARNING 05-10 10:15:38 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /workspace/envs/trimenders_v1/lib/python3.11/site-packages/vllm/model_executor/layers/quantization/utils/configs/N=6144,K=4096,device_name=NVIDIA_GeForce_RTX_4090,dtype=fp8_w8a8,block_shape=[128,128].json
(EngineCore pid=1891) WARNING 05-10 10:15:40 [fp8_utils.py:1185] Using default W8A8 Block FP8 kernel config. Performance might be sub-optimal! Config file not found at /workspace/envs/trimenders_v1/lib/py

# 2. 대상 비디오파일 전처리 (크롭, 프레임)

In [8]:
TEMP_CLIP = "/workspace/members/kim/temp_10sec_input.mp4"

ORIGIN_VIDEO = "/workspace/data/video/ai_hub_senior_anomaly/Validation/video/Abnormal_Behavior_Falldown/video_frames/H11H21H31/FD_In_H11H21H31_0009_20201229_14.mp4" 

START_TIME = 100      # 시작 시점 (초)
DURATION = 20         # 추출할 길이 (초)
FPS = 5.0             # 초당 프레임 수 (Sampling Rate)
WIDTH = 640           # 가로 해상도 (세로는 자동 비율)

# 계산된 정보 출력
total_frames = int(DURATION * FPS)
print(f"📊 설정 확인: {START_TIME}초 시점부터 {DURATION}초 동안 초당 {FPS}프레임 추출 (총 {total_frames}개 프레임)")

# 1. 원본에서 가공된 클립 추출
print(f"🎬 {START_TIME}초 시점부터 가공 중...")
cmd = [
    "ffmpeg", "-y", 
    "-ss", str(START_TIME),               # 입력 파일 탐색 속도를 위해 앞에 배치
    "-t", str(DURATION),                  # 추출 길이
    "-i", ORIGIN_VIDEO, 
    "-vf", f"fps={FPS},scale={WIDTH}:-1", # 변수화된 FPS와 해상도 적용
    "-vcodec", "libx264", 
    "-pix_fmt", "yuv420p", 
    "-loglevel", "error",
    TEMP_CLIP
]
try:
    subprocess.run(cmd, check=True)
    print(f"✅ 가공 완료: {TEMP_CLIP}")
    print(f"   (파일 크기: {os.path.getsize(TEMP_CLIP) / 1024 / 1024:.2f} MB)")
except subprocess.CalledProcessError as e:
    print(f"❌ 추출 실패: {e}")

📊 설정 확인: 100초 시점부터 20초 동안 초당 5.0프레임 추출 (총 100개 프레임)
🎬 100초 시점부터 가공 중...
✅ 가공 완료: /workspace/members/kim/temp_10sec_input.mp4
   (파일 크기: 0.48 MB)
time: 2.41 s (started: 2026-05-10 10:17:35 +00:00)


# 3. 모델 실행 (설명 텍스트 생성)

### 3-1) TIME_INTERVAL 변수값 설정

In [17]:
TIME_INTERVAL = 10

time: 428 μs (started: 2026-05-10 10:35:57 +00:00)


### 3-2) 프롬프트

In [26]:
# # (1) 낙상은 잘 감지하지만 불필요한 해석을 많이함.
# # 1. System Prompt
# system_prompt = """You are a professional Clinical Observation AI specializing in geriatric care and safety monitoring. 
# Your task is to analyze video clips of elderly individuals and provide a technical, objective log of their actions.

# [Strict Reporting Guidelines]
# 1. Precision of Movement: Use specific verbs for physical actions (e.g., 'staggering', 'shuffling', 'leaning', 'slumping').
# 2. No Euphemisms: Never use vague phrases like 'approaching the floor' or 'sitting down' if the person is falling. Use 'uncontrolled fall', 'collapse', or 'impact with ground'.
# 3. Environmental Interaction: Note if they are using assistive devices or if they are losing balance due to obstacles.
# 4. Anomaly Detection: Clearly flag any sudden changes in velocity or posture that indicate a medical emergency or a fall.
# """

# # 2. User Prompt
# user_prompt = f"""
# Analyze the provided video clip (approx. {DURATION} seconds).
# The recording starts at {START_TIME}s. Break down your observation into two 10-second intervals.

# ### [Clinical Activity Report]
# - Interval 1 ({START_TIME}.0s - {START_TIME + 10}.0s): 
# - Interval 2 ({START_TIME + 10}.0s - {START_TIME + 20}.0s): 

# [Summary of Concerns]: (Explicitly state if a fall, slip, or abnormal behavioral symptom was detected. If none, state 'Routine behavior observed.') 

# """

# # (2) 낙상을 잘 감지하지 못함.
# # 1. System Prompt
# system_prompt = """You are an objective video analysis AI. Describe only the visible physical actions and environmental facts.

# [Strict Constraints]
# 1. Visual Description Only: Describe only what is physically visible. Do not infer medical causes, internal states, or intentions.
# 2. No Interpretive Phrases: Avoid phrases like "appears to be disoriented," "signs of instability," or "suggests a cardiac event."
# 3. Factual Verbs: Use direct verbs such as 'standing', 'moving arms', 'falling', 'lying', 'motionless'.
# 4. No Contextual Speculation: Do not mention that something happened "without prior indication." Just state the action itself.
# """

# # 2. User Prompt
# user_prompt = f"""
# Analyze the provided video clip ({DURATION}s). 
# Provide a strictly factual description of the movements in each 10-second interval.

# ### [Activity Log]
# - Interval 1 ({START_TIME}.0s - {START_TIME + 10}.0s): 
# - Interval 2 ({START_TIME + 10}.0s - {START_TIME + 20}.0s): 

# [Current Status]: (State the final posture and movement status of the individual at the end of the clip.)
# """


time: 942 μs (started: 2026-05-10 10:39:54 +00:00)


In [ ]:
# # 1. System Prompt
# system_prompt = """
# """

# # 2. User Prompt
# user_prompt = f"""
# """


In [15]:
# <3>

# # 1. System Prompt
# system_prompt = """
# You are an objective video analysis AI. Describe physical actions with a focus on posture stability and movement velocity.

# [Strict Instructions]
# 1. Focus on Physical Dynamics: Specifically note any sudden changes in velocity (falling speed) and the loss of upright posture.
# 2. Stability Reporting: Explicitly state if the transition from standing to the floor is "controlled" (slow, intentional) or "uncontrolled" (rapid, sudden impact).
# 3. Posture Observation: Describe the alignment of the body during the transition. Use terms like "rapid descent" or "sudden collapse" if the movement is involuntary.
# 4. No Medical Inference: Do not guess the cause (e.g., stroke), but describe the effect (e.g., "the body impacts the floor rapidly").
# """

# # 2. User Prompt
# user_prompt = f"""
# Analyze the provided video clip ({DURATION}s). 
# Provide a factual log of the movements, specifically distinguishing between controlled transitions and uncontrolled falls.

# ### [Activity Log]
# - Interval 1 ({START_TIME}.0s - {START_TIME + 10}.0s): 
# - Interval 2 ({START_TIME + 10}.0s - {START_TIME + 20}.0s): 

# [Movement Quality]: (Was the descent to the floor controlled or uncontrolled? Describe the speed and impact.)
# [Current Status]: (Final posture and lack of movement.)
# """



time: 988 μs (started: 2026-05-10 10:33:48 +00:00)


In [34]:
# <4>
# 1. System Prompt
system_prompt = """
You are an objective video analysis AI. Your output must ONLY consist of the [Activity Log] section.

[Strict Constraints]
1. Factual Description Only: Describe physical movements, velocity, and balance. If the center of gravity shifts faster than a controlled sitting motion, specify it as a 'loss of balance' or 'sudden collapse'.
2. No Summary/Status: Do not provide "Current Status", "Summary of Concerns", or "Movement Quality" sections.
3. Describe the direction of the fall based on which body part makes contact with the floor first (e.g., knees, chest, side, or back). Do not assume the direction unless the movement trajectory is clearly visible.
4. No Inference: Describe the fall as an "uncontrolled rapid descent" if it lacks bracing motions, but do not guess the medical cause.
5. Output Format: Strictly follow the format below without any additional text.
"""

# 2. User Prompt
user_prompt = f"""
Analyze the provided video clip ({DURATION}s). 
Provide a factual description of the movements for each 10-second interval.

### [Activity Log]
- Interval 1 ({START_TIME}.0s - {START_TIME + 10}.0s): 
- Interval 2 ({START_TIME + 10}.0s - {START_TIME + 20}.0s):
"""


time: 1.25 ms (started: 2026-05-10 10:59:28 +00:00)


### 3-3) 메시지 구성, 파라미터 출력, 분석, 결과 출력

In [35]:
# 3. 메시지 구성
messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": TEMP_CLIP, "fps": FPS}, 
            {"type": "text", "text": user_prompt.strip()}
        ],
    }
]

# 4. 파라미터 설정 (중요: 온도를 높여 두 번째 문장 생성을 유도)
sampling_params = SamplingParams(
    temperature=0.3,         # 0.01은 너무 딱딱해서 첫 줄만 쓰고 멈출 수 있음
    max_tokens=512, 
    # top_p=0.9,
    repetition_penalty=1.2,  # 동일 문구 무한 반복 방지
    # presence_penalty=1.1,   # 새로운 관찰 내용 유도
    # stop=["###"]  # 리포트가 끝나면 정지
)

# 5. 분석 수행
print(f"🚀 Qwen3-VL 모델 분석 시작... (분석 구간: {START_TIME}s ~ {START_TIME + DURATION}s)(VRAM 사용량 및 추론 시간을 확인하세요)")
inputs = [prepare_inputs(messages, processor)]
outputs = llm.generate(inputs, sampling_params=sampling_params)

# 6. 결과 출력
print("\n" + "="*25 + " ANALYSIS REPORT " + "="*25)
print(f"📂 대상 파일: {os.path.basename(ORIGIN_VIDEO)}")
print(f"🎥 분석 설정: {START_TIME}s 시점부터 {DURATION}초간 (Sampling: {FPS}fps)")
print("-" * 67)
print(outputs[0].outputs[0].text.strip())
print("=" * 67)

🚀 Qwen3-VL 모델 분석 시작... (분석 구간: 100s ~ 120s)(VRAM 사용량 및 추론 시간을 확인하세요)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


========================= ANALYSIS REPORT =========================
📂 대상 파일: FD_In_H11H21H31_0009_20201229_14.mp4
🎥 분석 설정: 100s 시점부터 20초간 (Sampling: 5.0fps)
-------------------------------------------------------------------
[Activity Log]
- Interval 1 (0.0s - 10.0s):
The man stands in a doorway at the end of a hallway, adjusting his clothing while looking down. He then bends forward slightly before stepping backward onto a blue mat placed near the door frame.
- Interval 2 (10.0s - 20.0s):
He collapses face-first onto the mat, landing flat on his stomach. His arms remain extended outward beside him, and he remains still throughout this period.
time: 5.87 s (started: 2026-05-10 10:59:34 +00:00)
